In [1]:
import os
import re
import torch
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI

In [2]:
# Initialization
print("Initialize the RAG system...")

ENCODER_MODEL_PATH = "./GUI/all-MiniLM-L6-v2"
KNOWLEDGE_BASE_FILE = "./GUI/knowledge_base.txt"
API_KEY = "sk-2ca600b2a0be428a81dd5c94f6517d8b"
API_BASE = "https://api.deepseek.com"
API_MODEL = "deepseek-chat"

print("Load the encoding model...")
encoder = SentenceTransformer(ENCODER_MODEL_PATH)

print("Load the knowledge base...")
with open(KNOWLEDGE_BASE_FILE, 'r', encoding='utf-8') as f:
    docs = [line.strip() for line in f if line.strip()]
print(f"Knowledge base: {len(docs)} items")

print("Indexes are being built...")
doc_embeddings = encoder.encode(docs, convert_to_numpy=True)
dim = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(doc_embeddings)

print("Initialize the API client...")
client = OpenAI(api_key=API_KEY, base_url=API_BASE)

print("Initialization completed！")

Initialize the RAG system...
Load the encoding model...
Load the knowledge base...
Knowledge base: 5 items
Indexes are being built...
Initialize the API client...
Initialization completed！


In [3]:
# Analytic function
def parse_input_text(raw_text):
    lines = raw_text.strip().split('\n')
    
    data_line = None
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if all(keyword in line.lower() for keyword in ['sequence', 'label', 'structure']):
            continue
        if re.search(r'[AUGC]{3,}', line) and re.search(r'\d+\.\d+', line):
            data_line = line
            break
    
    if not data_line:
        return None
    
    fields = data_line.split('\t')
    fields = [f.strip() for f in fields if f.strip()]
    
    try:
        sequence = None
        for f in fields:
            if len(f) > 50 and re.match(r'^[AUGC\s]+$', f.replace(' ', '')):
                sequence = f
                break
        
        if not sequence:
            raise ValueError("Cannot find the sequence field")
        
        pure_numbers = []
        for f in fields:
            if ',' in f:
                continue
            if re.match(r'^-?\d+(\.\d+)?$', f.strip()):
                pure_numbers.append(f.strip())
        
        if len(pure_numbers) < 3:
            raise ValueError(f"Insufficient pure numeric fields: {len(pure_numbers)}")
        
        predicted_label = int(float(pure_numbers[-1]))
        predicted_prob = float(pure_numbers[-2])
        true_label = int(float(pure_numbers[-3]))
        
        return {
            'sequence': sequence,
            'true_label': true_label,
            'predicted_label': predicted_label,
            'predicted_prob': predicted_prob,
        }
        
    except Exception as e:
        print(f"Parsing error: {e}")
        return None

In [4]:
# RAG Function
def rag_explain(sequence, true_label, predicted_label, predicted_prob, top_k=2):
    # 1. Search the knowledge base
    query_text = f"RNA sequence: {sequence}"
    query_emb = encoder.encode([query_text], convert_to_numpy=True)
    scores, indices = index.search(query_emb, top_k)
    retrieved_docs = [docs[i] for i in indices[0]]
    
    # 2. Create prompt
    seq_short = str(sequence)[:50]
    pred_text = "binding" if predicted_label == 1 else "non-binding"
    true_text = "binding" if true_label == 1 else "non-binding"
    match_status = "CORRECT" if predicted_label == true_label else "INCORRECT"
   
    knowledge_text = "\n\n".join([
        f"Method {i+1}: {doc}" 
        for i, doc in enumerate(retrieved_docs)
    ])
    
    prompt = f"""You are an expert RNA biologist. Analyze this prediction and suggest validation experiments.

## Prediction Input
- RNA sequence (k-mers): {seq_short}
- Model prediction: {pred_text} (probability: {predicted_prob:.4f})
- True label: {true_text}
- Prediction status: {match_status}

## Retrieved Experimental Methods (from knowledge base)
{knowledge_text}

## Your Task
Based on the RNA sequence features and the retrieved experimental methods above, provide:

1. **Mechanistic Explanation**: Why does this RNA likely {pred_text} or not? Consider sequence motifs, structural features, or biological context. (2-3 sentences)

2. **Wet-lab Validation Plan**: Propose 2 specific experiments to validate this prediction. For each:
   - Which method to use (choose from retrieved methods above, or combine them)
   - Brief procedure outline
   - Expected outcome if the prediction is correct

Keep your response concise (under 150 words) and technically accurate."""

    # 3. Invoke LLM for generation
    try:
        response = client.chat.completions.create(
            model=API_MODEL,
            messages=[
                {"role": "system", "content": "You are an expert RNA biologist specializing in protein-RNA interactions."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=250,
            timeout=30
        )
        
        explanation = response.choices[0].message.content.strip()
        
        return {
            'status': 'success',
            'explanation': explanation,
            'retrieved_knowledge': retrieved_docs,  # Return to the original knowledge for reference
            'parsed_info': {
                'sequence_preview': seq_short,
                'true_label': true_text,
                'predicted': f"{pred_text} ({predicted_prob:.4f})",
                'match': match_status
            },
            'error': None
        }
        
    except Exception as e:
        return {
            'status': 'failed',
            'explanation': None,
            'retrieved_knowledge': retrieved_docs,
            'error': str(e)
        }

In [5]:
# Convenient function: Parsing + Generation
def rag_explain_from_text(raw_text):
    parsed = parse_input_text(raw_text)
    if not parsed:
        return {
            'status': 'failed',
            'error': 'The input text cannot be parsed.',
            'explanation': None,
            'retrieved_knowledge': [],
            'parsed_info': None
        }
    
    return rag_explain(**parsed)

In [6]:
# Formatted output
def format_result(result):
    if result['status'] != 'success':
        return f"Generation failed: {result['error']}"
    
    info = result['parsed_info']
    
    lines = [
        "RAG analysis results",
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━",
        f"Sequence: {info['sequence_preview']}...",
        f"True label: {info['true_label']}",
        f"Model prediction: {info['predicted']} [{info['match']}]",
        "",
        "The explanations and suggestions generated by LLM:",
        result['explanation'],
        "",
        "The retrieved knowledge base content:",
    ]
    
    for i, doc in enumerate(result['retrieved_knowledge'], 1):
        lines.append(f"  [{i}] {doc[:100]}...")
    
    lines.append("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
    
    return "\n".join(lines)

In [7]:
# Test 
if __name__ == "__main__":
    test_input = """sequence	label	structure	pairing_probabilities	predicted_probability	predicted_label
9	GCA CAC ACA CAU AUA UAG AGC GCA CAG AGG GGG GGG GGU GUU UUC UCA CAC ACU CUC UCC CCG CGU GUG UGA GAU AUA UAG AGG GGU GUA UAU AUU UUA UAU AUG UGA GAU AUG UGA GAU AUG UGA GAU AUA UAA AAU AUG UGU GUU UUC UCG CGG GGC GCC CCC CCU CUA UAA AAC ACA CAG AGC GCU CUG UGU GUC UCU CUG UGC GCC CCA CAC ACU CUG UGU GUG UGU GUU UUU UUC UCU CUG UGG GGU GUG UGA GAU AUG UGA GAG AGU GUU UUG UGG GGU GUU UUA UAG AGG	0	1,1,0,0,0,0,0,2,2,0,1,1,1,1,0,0,0,0,2,2,2,2,0,0,1,1,0,0,0,0,1,1,1,1,1,1,0,0,0,0,2,2,2,2,2,2,0,0,0,0,2,2,0,0,0,1,1,1,1,1,0,1,1,1,1,1,1,1,0,1,1,1,1,0,0,0,0,0,0,0,0,0,2,2,2,2,2,2,2,0,2,2,2,2,0,0,2,2,2,2,2	0.779,0.779,0.149,0.153,0.151,0.154,0.161,0.779,0.779,0.271,0.852,0.854,0.847,0.844,0.113,0.160,0.167,0.166,0.844,0.847,0.854,0.852,0.479,0.554,0.766,0.753,0.488,0.484,0.546,0.178,0.909,0.946,0.950,0.951,0.948,0.937,0.181,0.100,0.099,0.141,0.937,0.948,0.951,0.950,0.946,0.909,0.162,0.546,0.488,0.470,0.753,0.766,0.554,0.374,0.479,0.955,0.973,0.974,0.972,0.944,0.707,0.930,0.985,0.987,0.936,0.851,0.903,0.899,0.283,0.951,0.982,0.982,0.899,0.166,0.164,0.337,0.042,0.084,0.063,0.131,0.154,0.337,0.899,0.982,0.982,0.951,0.899,0.903,0.851,0.279,0.936,0.987,0.985,0.930,0.689,0.707,0.944,0.972,0.974,0.973,0.955	0.50593346	1"""
    
    result = rag_explain_from_text(test_input)
    print(format_result(result))

RAG analysis results
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Sequence: GCA CAC ACA CAU AUA UAG AGC GCA CAG AGG GGG GGG GG...
True label: non-binding
Model prediction: binding (0.5059) [INCORRECT]

The explanations and suggestions generated by LLM:
**Mechanistic Explanation:** The sequence is G-rich and contains multiple GGG repeats, which can form G-quadruplex structures. Many RBPs (e.g., FMRP, hnRNPs) specifically bind such motifs, making the model's prediction plausible despite the "non-binding" label.

**Wet-lab Validation Plan:**

1. **RNA pull-down:** Synthesize biotinylated RNA probe, incubate with cell lysate expressing the RBP of interest, pull down with streptavidin beads, and detect bound protein via western blot. *Expected outcome:* A protein band confirms binding.

2. **CLIP-seq variant (e.g., eCLIP):** Perform in cells expressing the RBP, followed by high-throughput sequencing. *Expected outcome:* A significant peak mapping to the RNA sequence confirms in vivo binding.

T